In [ ]:
import polars as pl

In [ ]:
DATA_PATH = "/kaggle/input/trendyol-e-ticaret-hackathonu-2025-kaggle/data"

train_sessions = pl.read_parquet(f"{DATA_PATH}/train_sessions.parquet")
test_sessions = pl.read_parquet(f"{DATA_PATH}/test_sessions.parquet")

content_metadata = pl.read_parquet(f"{DATA_PATH}/content/metadata.parquet")
content_price_data = pl.read_parquet(f"{DATA_PATH}/content/price_rate_review_data.parquet")

In [ ]:
train_sessions = train_sessions.with_columns(train_sessions["ts_hour"].cast(pl.Date).alias("ts_date"))
test_sessions = test_sessions.with_columns(test_sessions["ts_hour"].cast(pl.Date).alias("ts_date"))
content_price_data = content_price_data.with_columns(content_price_data["update_date"].cast(pl.Date).alias("ts_date"))

In [ ]:
train_sessions = train_sessions.join(content_metadata, on="content_id_hashed", how="left")
train_sessions = train_sessions.join(content_price_data, on=["content_id_hashed", "ts_date"], how="left")

test_sessions = test_sessions.join(content_metadata, on="content_id_hashed", how="left")
test_sessions = test_sessions.join(content_price_data, on=["content_id_hashed", "ts_date"], how="left")

In [ ]:
import lightgbm as lgb

train_data = train_sessions.select([
    "ordered", "content_review_count", "content_review_wth_media_count",
    "content_rate_count", "content_rate_avg", "attribute_type_count",
    "total_attribute_option_count", "merchant_count", "filterable_label_count",
    "original_price", "selling_price", "discounted_price"
    ])

train_X = train_data.drop("ordered").to_pandas()
train_y = train_data["ordered"].to_pandas()

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbose': -1,
}

model = lgb.LGBMClassifier(**params)
model.fit(train_X, train_y)

In [ ]:
test_sessions = test_sessions.with_columns(prediction = model.predict_proba(test_sessions.select(train_X.columns).to_pandas())[:, 1])
test_sessions = test_sessions.sort(["session_id", "prediction"], descending=True)

In [ ]:
submission_df = test_sessions.group_by("session_id").agg(
    pl.col("content_id_hashed").alias("prediction")
).with_columns(
    pl.col("prediction").list.join(" ")
)

In [ ]:
submission_df.write_csv("sample_submission_level1.csv")